In [ ]:
#training the model
#import libraries
import numpy as np
import cv2
from PIL import Image
import os
import PIL
import glob
import imutils
from matplotlib import pyplot as plt 

#Initialize
i = 0
waist = []
def is_hidden(file):
    return file.startswith('.')
# read files in loop
path_of_the_directory= 'Training images'

for filename in os.listdir(path_of_the_directory): #to access each image
    f = os.path.join(path_of_the_directory,filename)
    if is_hidden(f):
        print("hidden file")
    elif os.path.isfile(f) and not is_hidden(f):
        fixed_height = 400   
        image = Image.open(f)     
        #resizing the image maintaining the aspect ratio 
        height_percent = (fixed_height / float(image.size[1]))
        width_size = int((float(image.size[0]) * float(height_percent)))
        image = image.resize((width_size, fixed_height), PIL.Image.NEAREST)
        image.save('resized_front.jpg')
        img = cv2.imread('resized_front.jpg')
        img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB) #convert colour from BGR to RGB
         
        # define mask for grabcut method-creating numpy arrays
        mask = np.zeros(img.shape[:2],np.uint8)
        bgdModel = np.zeros((1,65),np.float64)
        fgdModel = np.zeros((1,65),np.float64)

        # define rectangle left, top, width, height
        rect = (100,130,100,350)

        # apply grabCut method to extract the foreground
        cv2.grabCut(img,mask,rect,bgdModel,fgdModel,20,cv2.GC_INIT_WITH_RECT)
        mask2 = np.where((mask==2)|(mask==0),0,1).astype('uint8')
        img = img*mask2[:,:,np.newaxis]
        cv2.imwrite('newimage.jpg',img)
        
        im = cv2.imread('newimage.jpg')
        imgray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        imgray = cv2.medianBlur(imgray, ksize=7)
        ret, thresh = cv2.threshold(imgray, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

        new = np.zeros(imgray.shape)
        new = cv2.drawContours(im,contours,len(contours)-1,(255,255,255),10)
        mask = np.zeros(imgray.shape,np.uint8)

        if len(contours) <= 0:
            print("No contours")
        else:     
            cv2.drawContours(mask,[contours[len(contours)-1]],0,255,-1)
            pixelpoints = cv2.findNonZero(mask)
            cv2.imwrite("masked_image.jpg",mask)
            #Masked image
            img = cv2.imread("masked_image.jpg")

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            ret, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY)
            cv2.bitwise_not(thresh, thresh)
            cnts = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                            cv2.CHAIN_APPROX_SIMPLE)
            cnts = cnts[0]
            c = max(cnts, key=cv2.contourArea)
            left = tuple(c[c[:, :, 0].argmin()][0])
            right = tuple(c[c[:, :, 0].argmax()][0])
            distance = np.sqrt( (right[0] - left[0])**2 + (right[1] - left[1])**2 )
            x,y,w,h = cv2.boundingRect(c)
            centx = np.sqrt( ((right[0] + left[0])**2)/4)
            centy = np.sqrt( ((right[1] + left[1])**2)/4 )
            font = cv2.FONT_HERSHEY_SIMPLEX
            cv2.circle(img, left, 5, (0, 0, 255), -1)
            cv2.circle(img, right, 5, (0, 0, 255), -1)
            cv2.circle(img, (int(centx), int(centy)), 5, (0, 0, 255), -1)
            cv2.line(img, left, right, (255,0,0), 2)
            cv2.drawContours(img, [c], -1, (0,255,0), 2)
            cv2.rectangle(img,(x,y),(x+w,y+h),(0,255,0),2)
    
            img = cv2.imread('masked_image.jpg')
            def find_marker(img):
                # convert the image to grayscale, blur it, and detect edges
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                gray = cv2.GaussianBlur(gray, (5, 5), 0)
                edged = cv2.Canny(gray, 35, 125)
                # find the contours in the edged image and keep the largest one;
                cnts = cv2.findContours(edged.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
                cnts = imutils.grab_contours(cnts)
                c = max(cnts, key = cv2.contourArea)
   
                # compute the bounding box of the of the image and return it
                mrker=cv2.minAreaRect(c)
                box = cv2.boxPoints(mrker)
                box = np.int64(box)
                pic=cv2.drawContours(edged,[box], 0, (0, 0, 255), 2)
                return cv2.minAreaRect(c)
            helo=find_marker(img)
     
            box = cv2.boxPoints(helo)
            box = np.int64(box)
            
            cv2.drawContours(img,[box], 0, (0, 0, 255), 1)
  
            x,y,w,h = box
          
            measure = w[0]-h[0]
            waist.append (measure)

